In [ ]:
#Installations and mounting to google drive

!pip install -q lxml python-docx

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Imports

import os
import json
import shutil
import hashlib
import logging
import zipfile
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from xml.etree import ElementTree as ET

from docx import Document
from docx.enum.text import WD_COLOR_INDEX
from tqdm.notebook import tqdm

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

In [ ]:
# =========================
# CONFIG
# =========================

ROOT_DIR = "/content/drive/MyDrive/ndtceda24"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/cards_output_processed"

USE_LOCAL_COPY = True
LOCAL_INPUT_DIR = "/content/ndtceda24"
LOCAL_WORK_DIR = "/content/cards_output_work"

N_WORKERS = 4
CHECKPOINT_EVERY_FILES = 100
SHARD_SIZE = 5000

CARD_START_STYLE_NAMES = {"heading 4"}
STOP_STYLE_NAMES = {"heading 1", "heading 2", "heading 3", "heading 4"}

CHECKPOINT_FILE = "checkpoint.json"
FAILED_FILE_LOG = "failed_files.txt"
SEEN_HASHES_FILE = "seen_hashes.json"

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

In [ ]:
# =========================
# XML / DOCX HELPERS
# =========================

W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
XML_NS = {"w": W_NS}

def qn(tag: str) -> str:
    prefix, local = tag.split(":")
    if prefix != "w":
        raise ValueError(f"Unsupported namespace prefix: {prefix}")
    return f"{{{W_NS}}}{local}"

# Converts a namespaced word xml tag like "w:p" into the full
# ElementTree namespace form "{namespace}p" so we can query XML safely

In [ ]:
def normalize_style_name(style_name: str) -> str:
    return " ".join((style_name or "").strip().lower().split())

In [ ]:
# =========================
# FILE DISCOVERY + STAGING
# =========================

def stage_root_to_local(src_root: str, dst_root: str, force_refresh: bool = False) -> str:
    src = Path(src_root)
    dst = Path(dst_root)

    if not src.exists():
        raise FileNotFoundError(f"Input root does not exist: {src}")

    if force_refresh and dst.exists():
        logger.info(f"Removing existing local copy: {dst}")
        shutil.rmtree(dst)

    if dst.exists():
        logger.info(f"Using existing local copy at {dst}")
        return str(dst)

    logger.info(f"Copying source tree to local SSD...\nFROM: {src}\nTO:   {dst}")
    shutil.copytree(src, dst)
    logger.info("Local copy complete.")
    return str(dst)


def discover_docx_files(root: str):
    root_path = Path(root)
    files = [
        p for p in sorted(root_path.rglob("*.docx"))
        if not p.name.startswith("~$")
    ]
    return files

In [ ]:
# =========================
# ATOMIC WRITE HELPERS
# =========================

def atomic_json_dump(obj, path: str, indent=None):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=indent)
    os.replace(tmp, path)


def atomic_text_write(text: str, path: str):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
    os.replace(tmp, path)

In [ ]:
# =========================
# CHECKPOINT + SEEN HASH PERSISTENCE
# =========================

# Checkpoint files let the pipeline resume after interruption
# instead of reparsing everything from scratch.

# Returns the local path where pipeline progress is stored
def checkpoint_path_local():
    return os.path.join(LOCAL_WORK_DIR, CHECKPOINT_FILE)

# Returns the local path where dedup hashes are stored
def seen_hashes_path_local():
    return os.path.join(LOCAL_WORK_DIR, SEEN_HASHES_FILE)

# Load prior pipeline progress if it exists, otherwise initialize defaults
def load_checkpoint():
    path = checkpoint_path_local()
    if not os.path.exists(path):
        return {
            "processed_files": [],
            "failed_files": [],
            "current_shard_index": 0,
            "total_unique_cards": 0,
        }

    with open(path, "r", encoding="utf-8") as f:
        state = json.load(f)

    state.setdefault("processed_files", [])
    state.setdefault("failed_files", [])
    state.setdefault("current_shard_index", 0)
    state.setdefault("total_unique_cards", 0)
    return state

# Persist current pipeline progress atomically
def save_checkpoint(state):
    atomic_json_dump(state, checkpoint_path_local(), indent=2)

# Load previously seen body hashes so duplicates are skipped across runs
def load_seen_hashes():
    path = seen_hashes_path_local()
    if not os.path.exists(path):
        return set()

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    return set(data)

# Persist dedup hashes atomically
def save_seen_hashes(seen_hashes):
    atomic_json_dump(sorted(seen_hashes), seen_hashes_path_local())

In [ ]:
# =========================
# STYLE MAP PARSING
# =========================

# Parses Word's styles.xml so we can map style IDs to human-readable names
# and follow "basedOn" inheritance when formatting is defined by parent styles.
def parse_styles_xml(styles_bytes: bytes):
    style_id_to_name = {}
    style_id_to_based_on = {}
    # style_id_to_name: maps internal Word style IDs -> visible style names
    # style_id_to_based_on: captures parent style inheritance chains

    if not styles_bytes:
        return style_id_to_name, style_id_to_based_on

    root = ET.fromstring(styles_bytes)

    for style in root.findall("w:style", XML_NS):
        style_id = style.get(qn("w:styleId"))
        if not style_id:
            continue

        name_el = style.find("w:name", XML_NS)
        based_on_el = style.find("w:basedOn", XML_NS)

        if name_el is not None:
            style_id_to_name[style_id] = name_el.get(qn("w:val"), style_id)
        else:
            style_id_to_name[style_id] = style_id

        if based_on_el is not None:
            base_id = based_on_el.get(qn("w:val"))
            if base_id:
                style_id_to_based_on[style_id] = base_id

    return style_id_to_name, style_id_to_based_on


def resolve_style_name(style_id, style_id_to_name, style_id_to_based_on, max_depth=10):
    if not style_id:
        return ""

    cur = style_id
    seen = set()

    for _ in range(max_depth):
        if cur in seen:
            break
        seen.add(cur)

        name = style_id_to_name.get(cur)
        if name:
            return name

        cur = style_id_to_based_on.get(cur)
        if not cur:
            break

    return style_id_to_name.get(style_id, style_id or "")

In [ ]:
# =========================
# XML TEXT / FORMAT HELPERS
# =========================

# Extract all visible text inside a single Word run (<w:r>)
# Some runs may contain multiple text nodes, so we join them.
def extract_text_from_run_xml(run_el):
    texts = []
    for t in run_el.findall(".//w:t", XML_NS):
        texts.append(t.text or "")
    return "".join(texts)


# Determines whether a run is underlined, highlighted, or emphasized
# by checking both direct run formatting and inherited style formatting.
def get_run_format_state_xml(run_el, style_id_to_name, style_id_to_based_on):
    """
    Returns (is_ul, is_hl, is_em)
    """
    rpr = run_el.find("w:rPr", XML_NS)

    is_ul = False
    is_hl = False
    is_em = False

    if rpr is not None:
        if rpr.find("w:u", XML_NS) is not None:
            is_ul = True

        if rpr.find("w:b", XML_NS) is not None:
            is_em = True

        hl = rpr.find("w:highlight", XML_NS)
        if hl is not None:
            hl_val = hl.get(qn("w:val"))
            if hl_val and hl_val.lower() not in {"none", "auto"}:
                is_hl = True

        rstyle = rpr.find("w:rStyle", XML_NS)
        if rstyle is not None:
            style_id = rstyle.get(qn("w:val"))
            style_name = resolve_style_name(style_id, style_id_to_name, style_id_to_based_on)
            if "Emphasis" in style_name:
                is_em = True

    return is_ul, is_hl, is_em


def paragraph_style_name_xml(p_el, style_id_to_name, style_id_to_based_on):
    ppr = p_el.find("w:pPr", XML_NS)
    if ppr is None:
        return ""

    pstyle = ppr.find("w:pStyle", XML_NS)
    if pstyle is None:
        return ""

    style_id = pstyle.get(qn("w:val"))
    return resolve_style_name(style_id, style_id_to_name, style_id_to_based_on)


def paragraph_text_xml(p_el):
    parts = []
    for node in p_el.findall(".//w:t", XML_NS):
        parts.append(node.text or "")
    return "".join(parts).strip()

In [ ]:
# =========================
# TAGGED PARAGRAPH RENDERING (XML FAST PATH)
# =========================

# Converts one paragraph into:
# 1. plain text with formatting stripped
# 2. tagged text where underline/highlight/emphasis are encoded as
#    <ul>, <h>, and <e> tags around the corresponding spans.
def render_paragraph_from_xml(p_el, style_id_to_name, style_id_to_based_on):
    plain_parts = []
    tagged_parts = []

    # Track whether each formatting state was active in the previous run
    # so tags only open/close when the formatting actually changes.
    prev_ul = prev_h = prev_e = False

    for child in p_el:
        if child.tag != qn("w:r"):
            continue

        text = extract_text_from_run_xml(child)
        if not text:
            continue

        is_ul, is_hl, is_em = get_run_format_state_xml(
            child, style_id_to_name, style_id_to_based_on
        )

        # Close tags that were active in the previous run but are no longer active now
        closing = ""
        if prev_e and not is_em:
            closing += "</e>"
        if prev_h and not is_hl:
            closing += "</h>"
        if prev_ul and not is_ul:
            closing += "</ul>"

        # Open tags for any formatting that begins on this run
        opening = ""
        if not prev_ul and is_ul:
            opening += "<ul>"
        if not prev_h and is_hl:
            opening += "<h>"
        if not prev_e and is_em:
            opening += "<e>"

        tagged_parts.append(closing + opening + text)
        plain_parts.append(text)

        prev_ul, prev_h, prev_e = is_ul, is_hl, is_em

    # Close any tags still left open at the end of the paragraph
    tail = ""
    if prev_e:
        tail += "</e>"
    if prev_h:
        tail += "</h>"
    if prev_ul:
        tail += "</ul>"
    tagged_parts.append(tail)

    return "".join(plain_parts).strip(), "".join(tagged_parts).strip()

In [ ]:
# =========================
# FAST XML PARSER (UPDATED WITH CITE/TEXT_BODY SPLIT)
# =========================

# Fast path parser: reads raw Word XML directly instead of using python-docx.
# This is much faster for large corpora and preserves formatting more reliably.
def fast_parse_docx(path: str):
    with zipfile.ZipFile(path, "r") as zf:
        if "word/document.xml" not in zf.namelist():
            raise ValueError("Missing word/document.xml")

        document_bytes = zf.read("word/document.xml")
        styles_bytes = zf.read("word/styles.xml") if "word/styles.xml" in zf.namelist() else b""

    style_id_to_name, style_id_to_based_on = parse_styles_xml(styles_bytes)
    doc_root = ET.fromstring(document_bytes)

    body = doc_root.find("w:body", XML_NS)
    if body is None:
        return []

    cards = []

    # State variables used while scanning the document paragraph-by-paragraph
    # and collecting content between a card-start style and a stop style.
    inside_card = False
    current_heading = None
    current_plain_parts = []
    current_tagged_parts = []

    # Finalizes the currently buffered card:
    # - joins paragraph text
    # - splits cite from body
    # - appends the structured training record
    def finalize_current():
        nonlocal current_heading, current_plain_parts, current_tagged_parts

        plain_text = "\n".join(x for x in current_plain_parts if x != "").strip()
        tagged_text = "\n".join(x for x in current_tagged_parts if x != "").strip()

        if plain_text:
            cite_plain, cite_formatted, body_plain, body_formatted = split_card_text(
                plain_text, tagged_text
            )

            cards.append({
                "heading": current_heading or "",
                "cite": cite_plain,
                "formatted_cite": cite_formatted,
                "body": body_plain,
                "formatted_body": body_formatted,
                "source_file": path,
            })

        current_heading = None
        current_plain_parts = []
        current_tagged_parts = []

    for child in body:
        if child.tag != qn("w:p"):
            continue

        style_name = normalize_style_name(
            paragraph_style_name_xml(child, style_id_to_name, style_id_to_based_on)
        )

        # A heading-style paragraph marks the beginning of a new card
        if style_name in CARD_START_STYLE_NAMES:
            if inside_card:
                finalize_current()

            current_heading = paragraph_text_xml(child)
            current_plain_parts = []
            current_tagged_parts = []
            inside_card = True
            continue

        if inside_card and style_name in STOP_STYLE_NAMES:
            finalize_current()
            inside_card = False
            continue

        if inside_card:
            plain_para, tagged_para = render_paragraph_from_xml(
                child, style_id_to_name, style_id_to_based_on
            )
            current_plain_parts.append(plain_para)
            current_tagged_parts.append(tagged_para)

    if inside_card:
        finalize_current()

    return cards

In [ ]:
# =========================
# FALLBACK PARSER (UPDATED WITH CITE/TEXT_BODY SPLIT)
# =========================

def run_flags_fallback(run):
    font = run.font
    style = run.style

    is_ul = bool(run.underline) or bool(font.underline)

    hl = font.highlight_color
    is_hl = hl is not None and hl not in (WD_COLOR_INDEX.AUTO, 0)

    is_em = bool(run.bold) or (style is not None and "Emphasis" in style.name)
    return is_ul, is_hl, is_em


def build_texts_fallback(body_paragraphs):
    plain_parts = []
    tagged_parts = []

    last_para_idx = len(body_paragraphs) - 1

    for para_idx, para in enumerate(body_paragraphs):
        prev_ul = prev_h = prev_e = False

        for run in para.runs:
            text = run.text
            if not text:
                continue

            is_ul, is_hl, is_em = run_flags_fallback(run)

            closing = ""
            if prev_e and not is_em:
                closing += "</e>"
            if prev_h and not is_hl:
                closing += "</h>"
            if prev_ul and not is_ul:
                closing += "</ul>"

            opening = ""
            if not prev_ul and is_ul:
                opening += "<ul>"
            if not prev_h and is_hl:
                opening += "<h>"
            if not prev_e and is_em:
                opening += "<e>"

            tagged_parts.append(closing + opening + text)
            plain_parts.append(text)

            prev_ul, prev_h, prev_e = is_ul, is_hl, is_em

        tail = ""
        if prev_e:
            tail += "</e>"
        if prev_h:
            tail += "</h>"
        if prev_ul:
            tail += "</ul>"
        tagged_parts.append(tail)

        if para_idx != last_para_idx:
            tagged_parts.append("\n")
            plain_parts.append("\n")

    return "".join(plain_parts).strip(), "".join(tagged_parts).strip()


def fallback_parse_docx(path: str):
    doc = Document(path)
    paragraphs = doc.paragraphs
    cards = []

    i = 0
    n = len(paragraphs)

    while i < n:
        para = paragraphs[i]
        style_name = normalize_style_name(para.style.name if para.style is not None else "")

        if style_name in CARD_START_STYLE_NAMES:
            heading = para.text.strip()
            i += 1

            body_paragraphs = []
            while i < n:
                p = paragraphs[i]
                p_style_name = normalize_style_name(p.style.name if p.style is not None else "")

                if p_style_name in STOP_STYLE_NAMES:
                    break

                body_paragraphs.append(p)
                i += 1

            plain_text, tagged_text = build_texts_fallback(body_paragraphs)

            if plain_text:
                cite_plain, cite_formatted, body_plain, body_formatted = split_card_text(
                    plain_text, tagged_text
                )

                cards.append({
                    "heading": heading,
                    "cite": cite_plain,
                    "formatted_cite": cite_formatted,
                    "body": body_plain,
                    "formatted_body": body_formatted,
                    "source_file": path,
                })
        else:
            i += 1

    return cards

In [ ]:
# =========================
# WORKER ENTRY
# =========================

# Worker-safe wrapper:
# tries the fast XML parser first, then falls back if needed,
# and always returns a structured result instead of crashing the pool.
def worker_parse_file(path: str):
def worker_parse_file(path: str):
    try:
        try:
            cards = fast_parse_docx(path)
            parser_used = "fast_xml"
        except Exception:
            cards = fallback_parse_docx(path)
            parser_used = "fallback"

        return {
            "path": path,
            "cards": cards,
            "error": None,
            "parser_used": parser_used,
        }

    except Exception as ex:
        return {
            "path": path,
            "cards": [],
            "error": str(ex),
            "parser_used": None,
        }

In [ ]:
# =========================
# DEDUP + SHARD WRITER (UPDATED TO NEW DATA PARSING FORMAT)
# =========================

# Normalize whitespace so visually identical bodies hash the same
def normalize_for_dedup(text: str) -> str:
    return " ".join(text.split()).strip()

# Deduplicate using only the training body text, not the heading/cite,
# so repeated evidence bodies are not duplicated in the dataset.
def dedup_key(text: str) -> str:
    normalized = normalize_for_dedup(text)
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


class ShardWriter:
    def __init__(self, out_dir_local: str, shard_size: int, start_index: int = 0):
        self.out_dir_local = out_dir_local
        self.shard_size = shard_size
        self.current_shard_index = start_index
        self.input_buffer = []
        self.output_buffer = []
        self.cards_in_current_shard = 0

    # input_buffer stores the plain-text training input
    # output_buffer stores the tagged target output
    def add_card(self, card):
        # INPUT format matches your uploaded input file
        self.input_buffer.append({
            "tag": card["heading"],
            "cite": card["cite"],
            "text_body": card["body"],
        })

        # OUTPUT format matches your uploaded output file
        self.output_buffer.append({
            "tag": card["heading"],
            "cite": card["cite"],
            "text_body": card["formatted_body"],
        })

        self.cards_in_current_shard += 1

        # Flush once the shard reaches the configured size
        if self.cards_in_current_shard >= self.shard_size:
            self.flush()

    def flush(self):
        if not self.input_buffer:
            return

        input_path = os.path.join(
            self.out_dir_local, f"input_{self.current_shard_index}.json"
        )
        output_path = os.path.join(
            self.out_dir_local, f"output_{self.current_shard_index}.json"
        )

        atomic_json_dump(self.input_buffer, input_path, indent=2)
        atomic_json_dump(self.output_buffer, output_path, indent=2)

        logger.info(
            f"Flushed shard {self.current_shard_index} with {len(self.input_buffer)} cards"
        )

        self.current_shard_index += 1
        self.input_buffer = []
        self.output_buffer = []
        self.cards_in_current_shard = 0

    def finalize(self):
        self.flush()

In [ ]:
# =========================
# CARD SPLITTING: HEADING / CITE / BODY
# =========================

# The first non-empty paragraph after a heading is treated as the citation.
# All later non-empty paragraphs become the actual evidence body.
def split_card_text(plain_text: str, formatted_text: str):
    """
    Split a card body into:
      - cite: first non-empty paragraph
      - text_body: remaining non-empty paragraphs

    Paragraph structure is preserved after the cite.
    """
    plain_paras = [p.strip() for p in plain_text.split("\n")]
    formatted_paras = [p.strip() for p in formatted_text.split("\n")]

    # keep aligned non-empty paragraphs only
    paired = [
        (p_plain, p_fmt)
        for p_plain, p_fmt in zip(plain_paras, formatted_paras)
        if p_plain.strip() != ""
    ]

    if not paired:
        return "", "", ""

    cite_plain = paired[0][0]
    cite_formatted = paired[0][1]

    remaining_plain = [x[0] for x in paired[1:]]
    remaining_formatted = [x[1] for x in paired[1:]]

    text_body_plain = "\n".join(remaining_plain).strip()
    text_body_formatted = "\n".join(remaining_formatted).strip()

    return cite_plain, cite_formatted, text_body_plain, text_body_formatted

In [ ]:
# =========================
# DRIVE SYNC
# =========================

# Sync locally written shard/checkpoint artifacts back to Drive
# so progress is preserved outside the Colab runtime.
def sync_local_outputs_to_drive():
    local_dir = Path(LOCAL_WORK_DIR)
    drive_dir = Path(DRIVE_OUTPUT_DIR)
    drive_dir.mkdir(parents=True, exist_ok=True)

    for p in local_dir.glob("input_*.json"):
        shutil.copy2(p, drive_dir / p.name)

    for p in local_dir.glob("output_*.json"):
        shutil.copy2(p, drive_dir / p.name)

    cp = local_dir / CHECKPOINT_FILE
    if cp.exists():
        shutil.copy2(cp, drive_dir / cp.name)

    ff = local_dir / FAILED_FILE_LOG
    if ff.exists():
        shutil.copy2(ff, drive_dir / ff.name)

    sh = local_dir / SEEN_HASHES_FILE
    if sh.exists():
        shutil.copy2(sh, drive_dir / sh.name)

In [ ]:
# =========================
# COMBINE ALL SHARDS INTO ONE BIG INPUT/OUTPUT
# =========================

# Merge all shard pairs into one full input/output dataset,
# while verifying that every input item still aligns with its output item.
# Ensure the same number of input and output shards exist
# Ensure shard indices line up
# Ensure the same number of examples exist in each aligned shard
# Ensure tag/cite fields still match between input and output records


def combine_all_shards(out_dir_local: str, drive_out_dir: str):
    local_dir = Path(out_dir_local)
    drive_dir = Path(drive_out_dir)

    input_files = sorted(local_dir.glob("input_*.json"), key=lambda p: int(p.stem.split("_")[1]))
    output_files = sorted(local_dir.glob("output_*.json"), key=lambda p: int(p.stem.split("_")[1]))

    if len(input_files) != len(output_files):
        raise ValueError(f"Mismatched shard counts: {len(input_files)} input vs {len(output_files)} output")

    all_inputs = []
    all_outputs = []

    for in_file, out_file in zip(input_files, output_files):
        in_idx = int(in_file.stem.split("_")[1])
        out_idx = int(out_file.stem.split("_")[1])

        if in_idx != out_idx:
            raise ValueError(f"Shard misalignment: {in_file.name} vs {out_file.name}")

        with open(in_file, "r", encoding="utf-8") as f:
            in_data = json.load(f)
        with open(out_file, "r", encoding="utf-8") as f:
            out_data = json.load(f)

        if len(in_data) != len(out_data):
            raise ValueError(f"Length mismatch in shard {in_idx}: {len(in_data)} vs {len(out_data)}")

        for j, (inp, out) in enumerate(zip(in_data, out_data)):
            if inp["tag"] != out["tag"] or inp["cite"] != out["cite"]:
                raise ValueError(
                    f"Alignment failure in shard {in_idx}, item {j}: "
                    f"input(tag/cite) != output(tag/cite)"
                )

        all_inputs.extend(in_data)
        all_outputs.extend(out_data)

    combined_input_path_local = local_dir / "input_full.json"
    combined_output_path_local = local_dir / "output_full.json"

    atomic_json_dump(all_inputs, str(combined_input_path_local), indent=2)
    atomic_json_dump(all_outputs, str(combined_output_path_local), indent=2)

    shutil.copy2(combined_input_path_local, drive_dir / "input_full.json")
    shutil.copy2(combined_output_path_local, drive_dir / "output_full.json")

    logger.info(f"Combined full input/output written with {len(all_inputs)} aligned examples")

In [ ]:
# =========================
# MAIN PIPELINE
# =========================

# End-to-end pipeline:
# 1. optionally stage files locally
# 2. discover .docx files
# 3. resume from checkpoint
# 4. parse files in parallel
# 5. deduplicate card bodies
# 6. write aligned shards
# 7. checkpoint periodically
# 8. sync results to Drive
# 9. combine shards into full datasets
def run_pipeline():
    active_root = ROOT_DIR
    if USE_LOCAL_COPY:
        active_root = stage_root_to_local(ROOT_DIR, LOCAL_INPUT_DIR, force_refresh=False)

    all_files = discover_docx_files(active_root)
    logger.info(f"Discovered {len(all_files)} .docx files")

    state = load_checkpoint()
    processed_files = set(state["processed_files"])
    failed_files = list(state["failed_files"])
    total_unique_cards = int(state["total_unique_cards"])
    start_shard_index = int(state["current_shard_index"])

    seen_hashes = load_seen_hashes()

    # Only submit files that have not already been marked processed in the checkpoint
    remaining_files = [str(p) for p in all_files if str(p) not in processed_files]
    logger.info(f"Already processed: {len(processed_files)}")
    logger.info(f"Remaining: {len(remaining_files)}")
    logger.info(f"Loaded persisted dedup hashes: {len(seen_hashes)}")

    shard_writer = ShardWriter(
        out_dir_local=LOCAL_WORK_DIR,
        shard_size=SHARD_SIZE,
        start_index=start_shard_index,
    )

    completed_since_checkpoint = 0
    fast_count = 0
    fallback_count = 0

    # Parse files in parallel so multiple .docx files are processed at once
    with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = [executor.submit(worker_parse_file, path) for path in remaining_files]

        # Consume results as workers finish, rather than waiting in submission order
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing files"):
            result = future.result()
            path = result["path"]

            if result["error"] is not None:
                failed_files.append({
                    "file": path,
                    "error": result["error"],
                })
            else:
                if result["parser_used"] == "fast_xml":
                    fast_count += 1
                elif result["parser_used"] == "fallback":
                    fallback_count += 1

                for card in result["cards"]:
                    # dedup on the actual training content body
                    key = dedup_key(card["body"])
                    if key not in seen_hashes:
                        seen_hashes.add(key)
                        shard_writer.add_card(card)
                        total_unique_cards += 1

            processed_files.add(path)
            completed_since_checkpoint += 1

            if completed_since_checkpoint >= CHECKPOINT_EVERY_FILES:
                shard_writer.flush()

                state = {
                    "processed_files": sorted(processed_files),
                    "failed_files": failed_files,
                    "current_shard_index": shard_writer.current_shard_index,
                    "total_unique_cards": total_unique_cards,
                }
                save_checkpoint(state)
                save_seen_hashes(seen_hashes)

                if failed_files:
                    failed_text = "\n".join(
                        f"{x['file']} :: {x['error']}" for x in failed_files
                    )
                    atomic_text_write(failed_text, os.path.join(LOCAL_WORK_DIR, FAILED_FILE_LOG))

                sync_local_outputs_to_drive()
                completed_since_checkpoint = 0

    shard_writer.finalize()

    final_state = {
        "processed_files": sorted(processed_files),
        "failed_files": failed_files,
        "current_shard_index": shard_writer.current_shard_index,
        "total_unique_cards": total_unique_cards,
    }
    save_checkpoint(final_state)
    save_seen_hashes(seen_hashes)

    if failed_files:
        failed_text = "\n".join(
            f"{x['file']} :: {x['error']}" for x in failed_files
        )
        atomic_text_write(failed_text, os.path.join(LOCAL_WORK_DIR, FAILED_FILE_LOG))

    sync_local_outputs_to_drive()
    combine_all_shards(LOCAL_WORK_DIR, DRIVE_OUTPUT_DIR)

    print("\nDone.")
    print(f"Unique cards written across persisted state: {total_unique_cards}")
    print(f"Persisted dedup hashes: {len(seen_hashes)}")
    print(f"Fast parser files: {fast_count}")
    print(f"Fallback parser files: {fallback_count}")
    print(f"Failed files: {len(failed_files)}")
    print(f"Output synced to: {DRIVE_OUTPUT_DIR}")
    print("Combined files: input_full.json and output_full.json")

In [ ]:
run_pipeline()

In [ ]:
# Quick inspection cell

from pathlib import Path

out_files = sorted(Path(DRIVE_OUTPUT_DIR).glob("input_*.json"))
print("Input shards:", len(out_files))

if out_files:
    sample_input = out_files[0]
    sample_output = Path(DRIVE_OUTPUT_DIR) / sample_input.name.replace("input_", "output_")

    with open(sample_input, "r", encoding="utf-8") as f:
        x = json.load(f)

    with open(sample_output, "r", encoding="utf-8") as f:
        y = json.load(f)

    print("\nSample input record:\n")
    print(json.dumps(x[0], indent=2, ensure_ascii=False))

    print("\nSample output record:\n")
    print(json.dumps(y[0], indent=2, ensure_ascii=False))

In [ ]:
# =========================
# ANALYZE FORMATTING DISTRIBUTION
# =========================

import json
from pathlib import Path

OUTPUT_PATH = "/content/drive/MyDrive/cards_output_Mich/output_full.json"  # change if needed

with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

total = len(data)

ul_count = 0
h_count = 0
e_count = 0

ul_only = 0
h_only = 0
e_only = 0

ul_h = 0
ul_e = 0
h_e = 0
all_three = 0
none = 0

for item in data:
    text = item["text_body"]

    has_ul = "<ul>" in text
    has_h = "<h>" in text
    has_e = "<e>" in text

    ul_count += has_ul
    h_count += has_h
    e_count += has_e

    if not (has_ul or has_h or has_e):
        none += 1
    elif has_ul and not has_h and not has_e:
        ul_only += 1
    elif has_h and not has_ul and not has_e:
        h_only += 1
    elif has_e and not has_ul and not has_h:
        e_only += 1
    elif has_ul and has_h and has_e:
        all_three += 1
    elif has_ul and has_h:
        ul_h += 1
    elif has_ul and has_e:
        ul_e += 1
    elif has_h and has_e:
        h_e += 1

def pct(x):
    return f"{(x/total*100):.2f}%" if total > 0 else "0%"

print("="*50)
print("TOTAL CARDS:", total)
print("="*50)

print("\n--- Individual Tag Presence ---")
print(f"Underline (<ul>): {ul_count} ({pct(ul_count)})")
print(f"Highlight (<h>): {h_count} ({pct(h_count)})")
print(f"Emphasis (<e>): {e_count} ({pct(e_count)})")

print("\n--- Exclusive Categories ---")
print(f"Only underline: {ul_only} ({pct(ul_only)})")
print(f"Only highlight: {h_only} ({pct(h_only)})")
print(f"Only emphasis: {e_only} ({pct(e_only)})")

print("\n--- Overlap Categories ---")
print(f"Underline + Highlight: {ul_h} ({pct(ul_h)})")
print(f"Underline + Emphasis: {ul_e} ({pct(ul_e)})")
print(f"Highlight + Emphasis: {h_e} ({pct(h_e)})")
print(f"All three: {all_three} ({pct(all_three)})")

print("\n--- No Formatting ---")
print(f"No tags: {none} ({pct(none)})")